### Create day datasets of 500 samples max for the large tweet dataset

In [ ]:
pip install whatthelang

In [ ]:
import numpy as np
import pandas as pd
from whatthelang import WhatTheLang

from tqdm import tqdm

import datetime
import os
from IPython.display import FileLink

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
df = pd.read_csv('/kaggle/input/bitcoin-tweets-20160101-to-20190329/tweets.csv', low_memory = False, delimiter=';', skiprows=0, lineterminator='\n', usecols = ['user', 'fullname','timestamp','text\r'])

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
# Create date column as time isn't required
def dt(x):
    t = pd.Timestamp(x)
    return pd.Timestamp.date(t)

df.timestamp = df.timestamp.apply(dt)

In [ ]:
df.head()

In [ ]:
# Remove old data
df= df[df['timestamp'] > datetime.date(2014, 1, 1)]

In [ ]:
df = df.sort_values(by=['timestamp'], ascending=True)

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.size

### limit number of tweets #01

In [ ]:
# limit number of tweets for each date to 500 due to computational limitations
df_grouped = df.groupby('timestamp').head(500)

In [ ]:
df_grouped.shape

In [ ]:
df_grouped['timestamp'].value_counts()

In [ ]:
df_grouped['timestamp'].unique().size

In [ ]:
os.makedirs('/kaggle/working/outputs')

In [ ]:
# Create checkpoint file
df_grouped.to_csv('/kaggle/working/outputs/tweets_large_filtered.csv')

In [ ]:
!zip -r file.zip './outputs'

In [ ]:
FileLink(r'file.zip')

### Remove null & non-english rows

In [ ]:
# Empty fullname is fine since user exists, remove row with empty user
df.dropna(subset=['user', 'timestamp', 'text\r'], inplace=True)

In [ ]:
df.size

only 4 rows have been removed... great news

In [ ]:
# Remove non-english text using WhatTheLang (https://github.com/indix/whatthelang)
wtl = WhatTheLang()
L = []

for row in tqdm(df['text\r']):
    if len(row) != 0:
        L.append(wtl.predict_lang(row))
    else:
        L.append(None)
        
df['lang'] = L
df.head()

In [ ]:
df_english = df[df['lang'] == 'en']

In [ ]:
df_english.head()

In [ ]:
df_english.drop(['lang'], axis=1, inplace=True)

In [ ]:
df_english.head()

### Limit number of tweets #02

In [ ]:
df_grouped_2 = df_english.groupby('timestamp').head(500)

In [ ]:
# Create checkpoint file
df_grouped_2.to_csv('/kaggle/working/outputs/tweets_large_filtered_cleaned.csv')

In [ ]:
!zip -r file.zip './outputs'
FileLink(r'file.zip')

### Create day datasets (use if necessary)

In [ ]:
df_days = [y for x, y in df_grouped_2.groupby('timestamp')]

In [ ]:
df_days[0]

In [ ]:
df_days[len(df_days) - 1]

In [ ]:
len(df_days)

In [ ]:
str(df_days[len(df_days) - 1].iloc[0]['timestamp'])

In [ ]:
# Create individual CSVs for each day
for i in tqdm(df_days):
    filename = str(i.iloc[0]['timestamp'])
    i.to_csv(f'/kaggle/working/outputs/{filename}.csv')

In [ ]:
!zip -r file.zip './outputs'
FileLink(r'file.zip')